# ProGen2 Protein Sequence Generation and Scoring

![ProGen2 Protein Sequence Generation and Scoring](https://proto-bio.github.io/proto-assets/images/tool/progen2/hero.png)

This notebook demonstrates both functions of the ProGen2 tool. In the first section, we use `run_progen2_sample` to generate protein sequences from amino acid prompts using autoregressive sampling. In the second section, we use `run_progen2_score` to compute log-likelihood and perplexity metrics that quantify how natural a given sequence appears to the model, enabling ranking and filtering of design candidates.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("progen2")
display_overview("progen2")
display_docs_section("progen2", "Background")

# ProGen2

ProGen2 is Salesforce's autoregressive protein language model for de novo protein sequence generation and scoring. Unlike masked language models (ESM2/ESM3) that predict masked positions bidirectionally, ProGen2 generates proteins left-to-right from a prompt and provides autoregressive likelihood scoring. The tool supports local GPU execution via a standalone venv .

## Background

[Autoregressive](https://en.wikipedia.org/wiki/Autoregressive_model) protein language models like ProGen2 learn the statistical patterns of natural protein sequences from large databases ([UniRef](https://www.uniprot.org/help/uniref), BFD, [OAS](https://opig.stats.ox.ac.uk/webapps/oas/)). By training to predict each amino acid given the preceding context, the model implicitly captures local motifs, long-range dependencies (such as distal residue co-evolution), and sequence patterns characteristic of particular protein families

This learned distribution enables two key applications: **generation** (sampling new sequences that follow natural protein statistics) and **scoring** (evaluating how "protein-like" a given sequence is under the model). Lower perplexity indicates a sequence is more consistent with the model's learned distribution of natural proteins.

## Available tools

In [2]:
display_available_tools("progen2")

- **`run_progen2_sample()`** — Sample protein sequences using ProGen2 language model
- **`run_progen2_score()`** — Score protein sequences using ProGen2 language model

### `run_progen2_sample`

ProGen2 generates protein sequences autoregressively from a prompt, extending the sequence one residue at a time according to its learned distribution over natural proteins. Available checkpoints range from 151M to 6B parameters, with a specialized OAS variant for antibody sequences. The `temperature` and `top_p` parameters control sampling diversity: lower temperature produces more conservative, high-confidence extensions while higher values explore more of sequence space.

In [3]:
from proto_tools import (
    ProGen2SampleInput,
    ProGen2SampleConfig,
    run_progen2_sample,
)

In [4]:
# Display docs
display_api_reference("progen2", "input", "run_progen2_sample")

# Create a ProGen2 input with an N-terminal amino acid prompt
inputs = ProGen2SampleInput(prompts=["MKTAYIAKQRQISF"])

**Input** — `CausalModelSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `prompts` | `list[str]` | required | Prompt sequence(s) to condition generation on |

In [5]:
# Display config docs
display_api_reference("progen2", "config", "run_progen2_sample")

# Configure sampling parameters | see docs above for all fields
config = ProGen2SampleConfig(
    model_checkpoint="progen2-large",
    max_new_tokens=120,
    temperature=0.2,
    top_p=0.95,
    top_k=0,
    truncate_at_stop=True,
    strip_special_tokens=True,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ProGen2SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `prepend_prompt` | `bool` | `True` | Include the input prompt at the start of each generated sequence |
| `temperature` | `float` | `0.2` | Softmax temperature for sampling; lower is more deterministic |
| `top_p` | `float` | `0.95` | Nucleus sampling threshold over per-position token probabilities |
| `batch_size` | `int` | `1` | Prompts per GPU forward pass; raise for throughput, lower if OOM |
| `model_checkpoint` | `Literal['progen2-small', 'progen2-medium', 'progen2-base', 'progen2-oas', 'progen2-large', 'progen2-BFD90', 'progen2-xlarge']` | `'progen2-large'` | ProGen2 weights variant |
| `local_path` | `str \| None` | `None` | Override the default download with a local weights directory |
| `top_k` | `int` | `0` | Top-k truncation; 0 disables and uses top-p only |
| `max_new_tokens` | `int` | `256` | Maximum number of new tokens to generate per prompt |
| `truncate_at_stop` | `bool` | `True` | Truncate generated sequences at the first stop token |
| `strip_special_tokens` | `bool` | `True` | Strip ProGen2 start/stop sentinel tokens from output |
| `return_logits` | `bool` | `False` | Include per-position logits in the output (large; disable to save memory) |

In [6]:
# Run the sampling function
sample_result = run_progen2_sample(inputs, config)

Running run_progen2_sample [00:00]

In [7]:
# Display output docs
display_api_reference("progen2", "output", "run_progen2_sample")

# Show the prompt and the generated continuation
prompt = inputs.prompts[0]
generated = sample_result.sequences[0]
print(f"Prompt:       {prompt}")
print(f"Generated:    {generated[:80]}..." if len(generated) > 80 else f"Generated:    {generated}")
print(f"Total length: {len(generated)} residues")

**Output** — `ProGen2SampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Generated sequences |
| `logits` | `list[list[list[float]]] \| None` | `None` | Per-position logits for each generated sequence |

Prompt:       MKTAYIAKQRQISF
Generated:    MKTAYIAKQRQISFEELLELAKKQEEIPEKKDKQRNVIRMKAQLTRSKTIDKIEELKEREEELERLQVTTIQEIRRMHNK...
Total length: 120 residues


### `run_progen2_score`

ProGen2 scores sequences by computing the autoregressive log-likelihood at each position, measuring how predictable each residue is given its preceding context. The resulting perplexity summarizes sequence naturalness: lower perplexity indicates the sequence more closely matches the statistical patterns of natural proteins in the training data. This is useful for ranking designed candidates, filtering out sequences unlikely to fold or function, or comparing wild-type and mutant variants.

In [8]:
from proto_tools import (
    ProGen2ScoringInput,
    ProGen2ScoringConfig,
    run_progen2_score,
)

In [9]:
# Display docs
display_api_reference("progen2", "input", "run_progen2_score")

# Score two protein sequences
score_inputs = ProGen2ScoringInput(sequences=["MVLSPADKTN", "MKTLLILAVVAA"])

**Input** — `CausalModelScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Sequences to score |

In [10]:
# Display config docs
display_api_reference("progen2", "config", "run_progen2_score")

# Configure scoring | see docs above for all fields
score_config = ProGen2ScoringConfig(
    batch_size=2,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ProGen2ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `batch_size` | `int` | `1` | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| `return_logits` | `bool` | `False` | Include per-position logits in the output (large; disable to save memory) |
| `model_checkpoint` | `Literal['progen2-small', 'progen2-medium', 'progen2-base', 'progen2-oas', 'progen2-large', 'progen2-BFD90', 'progen2-xlarge']` | `'progen2-large'` | ProGen2 weights variant |
| `local_path` | `str \| None` | `None` | Override the default download with a local weights directory |

In [11]:
# Run the scoring function
score_result = run_progen2_score(score_inputs, score_config)

Running run_progen2_score [00:00]

In [12]:
# Display output docs
display_api_reference("progen2", "output", "run_progen2_score")

# Show scoring results for each input sequence
for i, seq in enumerate(score_inputs.sequences):
    score = score_result.scores[i]
    print(f"Sequence {i + 1}: {seq}")
    print(f"    Log-likelihood:     {score.log_likelihood:.3f}")
    print(f"    Avg log-likelihood: {score.avg_log_likelihood:.3f}")
    print(f"    Perplexity:         {score.perplexity:.3f}")

**Output** — `CausalModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `list[proto_tools.tools.causal_models.shared_data_models.CausalModelScoringMetrics]` | required | List of scoring outputs, one per input sequence |

Sequence 1: MVLSPADKTN
    Log-likelihood:     -26.635
    Avg log-likelihood: -2.664
    Perplexity:         14.346
Sequence 2: MKTLLILAVVAA
    Log-likelihood:     -24.403
    Avg log-likelihood: -2.034
    Perplexity:         7.641


## Export Results

Generated sequences and scoring metrics can be saved to standard file formats for downstream use. Sampling results support FASTA, TXT, and JSON export. Scoring results support CSV and JSON export.

In [13]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sample_result.export(name="progen2_sequences", export_path=output_dir, file_format="fasta")
print(f"Generated sequences exported to {output_dir / 'progen2_sequences.fasta'}")

score_result.export(name="progen2_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'progen2_scores.csv'}")

Generated sequences exported to example_output/progen2_sequences.fasta
Scores exported to example_output/progen2_scores.csv
